# 13 — Sentence & Document Embeddings

**Learning objective.** Create fixed-length document vectors and use them for semantic comparison.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
texts=[
 'reset my forgotten password','cannot sign into my account','charged twice on my card',
 'duplicate payment on statement','change profile email address','update account contact details']
vec=TfidfVectorizer(ngram_range=(1,2))
X=vec.fit_transform(texts)
svd=TruncatedSVD(n_components=4,random_state=42)
Z=Normalizer().fit_transform(svd.fit_transform(X))
from sklearn.metrics.pairwise import cosine_similarity
sim=cosine_similarity(Z)
pd.DataFrame(np.round(sim,2),index=texts,columns=[f'd{i}' for i in range(len(texts))])

                                  d0    d1    d2    d3   d4    d5
reset my forgotten password     1.00  0.46  0.46 -0.27  0.0 -0.27
cannot sign into my account     0.46  1.00  0.31 -0.09 -0.0  0.73
charged twice on my card        0.46  0.31  1.00  0.73  0.0 -0.09
duplicate payment on statement -0.27 -0.09  0.73  1.00 -0.0  0.03
change profile email address    0.00 -0.00  0.00 -0.00  1.0  0.00
update account contact details -0.27  0.73 -0.09  0.03  0.0  1.00

`TF-IDF → SVD` is a classical dense document embedding baseline. Modern sentence-transformer embeddings are usually stronger for semantic similarity, but this baseline is transparent, fast, and fully offline.

---
    ## Production takeaways
    - Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
    - Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
    - Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- Distinguish token embeddings from document embeddings
- Use cosine similarity for semantic comparison